
# 06 — Multi-Source Customer Support: Retrieval Tools and Routing

**Course adaptation:** `multi_source_customer_support_agent.ipynb`
and `introduction_to_openai_ai_agents_sdk.ipynb` from
[BrightPool's Prompt Engineering Course](https://github.com/BrightPool/udemy-prompt-engineering-course).

**Goal:** Combine structured customer/order data stored in
SQLite with unstructured Markdown policy documents using
separate retrieval tools.


This notebook first implements and tests retrieval tools
without an LLM, then connects them to Gemini for live
tool selection and grounded response generation.

**API usage:** Offline tests require 0 Gemini/OpenAI requests.
The live agent demonstration uses 2 Gemini requests.



## 1. Why Multiple Data Sources?

A customer-support system may need information from
different sources.

**Structured data:** Orders, customers, products, and order
statuses stored in relational database tables.

**Unstructured data:** Return policies, shipping guidelines,
and warranty documents written as text.

We'll implement three tools:

1. `search_orders` — Retrieve customer orders from SQLite.
2. `search_products` — Retrieve products from SQLite.
3. `search_knowledge_base` — Search policy documents.

The tools use predefined operations rather than executing
arbitrary SQL generated by an AI model.


In [10]:

import sqlite3
import re
import json

conn = sqlite3.connect(":memory:")
conn.row_factory = sqlite3.Row

conn.executescript("""
CREATE TABLE customers (
    id INTEGER PRIMARY KEY,
    name TEXT,
    email TEXT UNIQUE
);

CREATE TABLE products (
    id INTEGER PRIMARY KEY,
    name TEXT,
    category TEXT,
    price REAL,
    in_stock INTEGER
);

CREATE TABLE orders (
    id INTEGER PRIMARY KEY,
    customer_id INTEGER,
    product_id INTEGER,
    status TEXT,
    FOREIGN KEY(customer_id) REFERENCES customers(id),
    FOREIGN KEY(product_id) REFERENCES products(id)
);
""")

conn.executemany(
    "INSERT INTO customers VALUES (?, ?, ?)",
    [
        (1, "Maya Chen", "maya@example.test"),
        (2, "Noah Patel", "noah@example.test")
    ]
)

conn.executemany(
    "INSERT INTO products VALUES (?, ?, ?, ?, ?)",
    [
        (101, "Wireless Earbuds", "audio", 39.99, 1),
        (102, "Bluetooth Speaker", "audio", 59.99, 1),
        (103, "Travel Headphones", "audio", 89.99, 0)
    ]
)

conn.executemany(
    "INSERT INTO orders VALUES (?, ?, ?, ?)",
    [
        (1001, 1, 101, "shipped"),
        (1002, 1, 102, "delivered"),
        (1003, 2, 103, "pending")
    ]
)

conn.commit()

print(
    "Customers:",
    conn.execute("SELECT COUNT(*) FROM customers").fetchone()[0]
)
print(
    "Orders:",
    conn.execute("SELECT COUNT(*) FROM orders").fetchone()[0]
)


Customers: 2
Orders: 3



## 2. Customer Support Tools

Implement three restricted retrieval functions:

- `search_orders`: Find orders by order ID or customer email.
- `search_products`: Filter products by category and stock.
- `search_knowledge_base`: Search policy documents using keywords.

Each tool returns factual data for a potential AI agent to use.


In [11]:

# Synthetic policy documents

KB = {
    "returns.md": (
        "Returns: Unused items can be returned within "
        "30 days of delivery with proof of purchase. "
        "Opened earbuds are not eligible for return."
    ),

    "shipping.md": (
        "Shipping: Standard shipping usually takes "
        "3 to 5 business days. "
        "Tracking is provided after dispatch."
    ),

    "warranty.md": (
        "Warranty: Electronics have a one-year limited "
        "warranty covering manufacturing defects."
    )
}


def search_orders(order_id=None, customer_email=None):
    if order_id is None and customer_email is None:
        raise ValueError(
            "Specify order_id or customer_email"
        )

    sql = """
        SELECT
            o.id,
            c.name AS customer,
            p.name AS product,
            o.status
        FROM orders o
        JOIN customers c ON o.customer_id = c.id
        JOIN products p ON o.product_id = p.id
        WHERE 1=1
    """

    params = []

    if order_id is not None:
        if not isinstance(order_id, int):
            raise TypeError("order_id must be an integer")

        sql += " AND o.id = ?"
        params.append(order_id)

    if customer_email is not None:
        sql += " AND c.email = ?"
        params.append(customer_email)

    return [
        dict(row)
        for row in conn.execute(sql, params).fetchall()
    ]


def search_products(category=None, in_stock_only=False):
    sql = """
        SELECT id, name, category, price, in_stock
        FROM products
        WHERE 1=1
    """

    params = []

    if category is not None:
        sql += " AND category = ?"
        params.append(category)

    if in_stock_only:
        sql += " AND in_stock = 1"

    return [
        dict(row)
        for row in conn.execute(sql, params).fetchall()
    ]


def search_knowledge_base(query):
    stopwords = {
        "the", "a", "is", "my", "i",
        "what", "about", "can", "do"
    }

    terms = (
        set(re.findall(r"[a-z]+", query.lower()))
        - stopwords
    )

    if not terms:
        return []

    ranked = []

    for name, content in KB.items():
        document_words = re.findall(
            r"[a-z]+",
            content.lower()
        )

        hits = len(
            terms.intersection(document_words)
        )

        if hits:
            ranked.append({
                "source": name,
                "matching_terms": hits,
                "text": content
            })

    return sorted(
        ranked,
        key=lambda row: (
            -row["matching_terms"],
            row["source"]
        )
    )


print("Order:", search_orders(order_id=1001))

print(
    "Available audio:",
    search_products(
        category="audio",
        in_stock_only=True
    )
)

print(
    "Policy:",
    search_knowledge_base(
        "Can I return unused items within 30 days?"
    )
)


Order: [{'id': 1001, 'customer': 'Maya Chen', 'product': 'Wireless Earbuds', 'status': 'shipped'}]
Available audio: [{'id': 101, 'name': 'Wireless Earbuds', 'category': 'audio', 'price': 39.99, 'in_stock': 1}, {'id': 102, 'name': 'Bluetooth Speaker', 'category': 'audio', 'price': 59.99, 'in_stock': 1}]
Policy: [{'source': 'returns.md', 'matching_terms': 5, 'text': 'Returns: Unused items can be returned within 30 days of delivery with proof of purchase. Opened earbuds are not eligible for return.'}, {'source': 'shipping.md', 'matching_terms': 1, 'text': 'Shipping: Standard shipping usually takes 3 to 5 business days. Tracking is provided after dispatch.'}]



## 3. Tool Routing

An AI agent needs a mechanism for selecting and executing
tools based on a user's request.

The original course uses an LLM to select tools.

Here, we explicitly supply tool names and arguments to test
the dispatch mechanism without making model API requests.

**Important:** This demonstrates deterministic tool routing,
not autonomous AI tool selection.


In [12]:

TOOLS = {
    "search_orders": search_orders,
    "search_products": search_products,
    "search_knowledge_base": search_knowledge_base
}


def dispatch_tool(name, arguments):
    if name not in TOOLS:
        raise ValueError("Unapproved tool")

    if not isinstance(arguments, dict):
        raise TypeError(
            "Arguments must be a dictionary"
        )

    return TOOLS[name](**arguments)


requests_to_test = [
    (
        "search_orders",
        {"order_id": 1001}
    ),
    (
        "search_products",
        {
            "category": "audio",
            "in_stock_only": True
        }
    ),
    (
        "search_knowledge_base",
        {
            "query": "return policy unused items 30 days"
        }
    )
]

for name, args in requests_to_test:
    result = dispatch_tool(name, args)

    print(
        name,
        json.dumps(result, indent=2)
    )


search_orders [
  {
    "id": 1001,
    "customer": "Maya Chen",
    "product": "Wireless Earbuds",
    "status": "shipped"
  }
]
search_products [
  {
    "id": 101,
    "name": "Wireless Earbuds",
    "category": "audio",
    "price": 39.99,
    "in_stock": 1
  },
  {
    "id": 102,
    "name": "Bluetooth Speaker",
    "category": "audio",
    "price": 59.99,
    "in_stock": 1
  }
]
search_knowledge_base [
  {
    "source": "returns.md",
    "matching_terms": 4,
    "text": "Returns: Unused items can be returned within 30 days of delivery with proof of purchase. Opened earbuds are not eligible for return."
  },
  {
    "source": "shipping.md",
    "matching_terms": 1,
    "text": "Shipping: Standard shipping usually takes 3 to 5 business days. Tracking is provided after dispatch."
  }
]



## 4. Validation and Tests

Verify that the tools:

- Retrieve the correct order status.
- Return an empty list for unknown orders.
- Filter available products correctly.
- Find the relevant return-policy document.
- Reject unknown tool names.

These are offline tests. A complete customer-support
agent would also need model-based tool selection,
grounded answer generation, and authorization checks
before accessing private customer records.


In [13]:

# Test known order
assert search_orders(order_id=1001)[0]["status"] == "shipped"

# Test unknown order
assert search_orders(order_id=9999) == []

# Test available audio products
assert len(
    search_products(
        category="audio",
        in_stock_only=True
    )
) == 2

# Test return policy retrieval
assert (
    search_knowledge_base(
        "unused items return 30 days"
    )[0]["source"] == "returns.md"
)

# Test rejection of unsupported tools
try:
    dispatch_tool(
        "delete_order",
        {"order_id": 1001}
    )

except ValueError:
    print("Unknown tool correctly rejected")

else:
    raise AssertionError(
        "Unknown tool was accepted"
    )

print("All offline tool tests passed.")
print("Gemini API requests: 0")


Unknown tool correctly rejected
All offline tool tests passed.
Gemini API requests: 0



## 5. Gemini-Powered Customer Support Agent

Connect the verified retrieval tools to Gemini using
the OpenAI-compatible Chat Completions API.

The workflow:
1. Gemini receives a customer question and available tool definitions.
2. Gemini requests the tools it considers necessary.
3. Python executes the approved tools.
4. Tool results are added to the conversation.
5. Gemini generates a grounded answer.

The demonstration uses synthetic customer data.

For this test, a complete answer requires information
from both `search_orders` and `search_knowledge_base`.

The workflow uses at most two Gemini requests and
does not automatically retry failed requests.


In [17]:
from openai import OpenAI
from getpass import getpass

MODEL = "gemini-3.5-flash-lite"

if "client" not in globals():
    gemini_key = getpass("Enter Gemini API key: ")

    client = OpenAI(
        api_key=gemini_key,
        base_url="https://generativelanguage.googleapis.com/v1beta/openai/",
        max_retries=0
    )

safe_client = client.with_options(
    max_retries=0,
    timeout=120.0
)
print("Gemini client ready.")

Gemini client ready.


In [18]:

agent_tools = [
    {
        "type": "function",
        "function": {
            "name": "search_orders",
            "description": "Look up an order using its order ID.",
            "parameters": {
                "type": "object",
                "properties": {
                    "order_id": {
                        "type": "integer",
                        "description": "Order number"
                    }
                },
                "required": ["order_id"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_products",
            "description": "Search the product catalogue.",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {"type": "string"},
                    "in_stock_only": {"type": "boolean"}
                }
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "search_knowledge_base",
            "description": "Find shipping, returns, and warranty policies.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "Keywords or question about a policy"
                    }
                },
                "required": ["query"]
            }
        }
    }
]

print("Tools available to Gemini:", [
    tool["function"]["name"] for tool in agent_tools
])


Tools available to Gemini: ['search_orders', 'search_products', 'search_knowledge_base']


In [22]:


history = [
    {
        "role": "system",
        "content": (
            "You are a customer-support assistant. "
            "Use tools to retrieve factual information. "
            "Do not invent order statuses or policies."
        )
    },
    {
        "role": "user",
        "content": (
            "What is the status of order 1001, "
            "and what is your return policy? "
            "Use both the order lookup and knowledge-base tools."
        )
    }
]

first = safe_client.chat.completions.create(
    model=MODEL,
    messages=history,
    tools=agent_tools,
    tool_choice="required"
)

assistant_message = first.choices[0].message
calls = assistant_message.tool_calls or []

print("Tools requested:", [c.function.name for c in calls])

if not calls:
    print("No tools requested; cannot complete the demonstration.")

else:
    history.append(assistant_message.model_dump(exclude_none=True))

    executed_tools = set()

    for call in calls:
        name = call.function.name
        arguments = json.loads(call.function.arguments)

        result = dispatch_tool(name, arguments)
        executed_tools.add(name)

        print("\nTool:", name)
        print("Result:", result)

        history.append({
            "role": "tool",
            "tool_call_id": call.id,
            "content": json.dumps(result)
        })

    required_tools = {
        "search_orders",
        "search_knowledge_base"
    }

    if not required_tools.issubset(executed_tools):
        print(
            "\nIncomplete: Gemini did not request both "
            "required tools. No additional API request made."
        )

    else:
        second = safe_client.chat.completions.create(
            model=MODEL,
            messages=history
        )

        print(
            "\nFinal answer:",
            second.choices[0].message.content
        )


Tools requested: ['search_orders', 'search_knowledge_base']

Tool: search_orders
Result: [{'id': 1001, 'customer': 'Maya Chen', 'product': 'Wireless Earbuds', 'status': 'shipped'}]

Tool: search_knowledge_base
Result: [{'source': 'returns.md', 'matching_terms': 1, 'text': 'Returns: Unused items can be returned within 30 days of delivery with proof of purchase. Opened earbuds are not eligible for return.'}]

Final answer: The status of order 1001 (Wireless Earbuds) is **shipped**.

Regarding our return policy: Unused items can be returned within 30 days of delivery with proof of purchase. Please note that opened earbuds are not eligible for return.



## Findings and Limitations

This notebook demonstrates multi-source customer support
using SQLite retrieval, keyword-based document search,
and live Gemini function calling.

### Findings

- SQLite tools retrieved structured order and product information.
- Knowledge-base search retrieved the relevant return policy.
- Offline tests passed, including rejection of unknown tool names.
- Gemini 3.5 Flash Lite selected both `search_orders` and
  `search_knowledge_base` for a single customer question.
- Python executed both tools and returned their results to Gemini.
- Gemini generated a final answer consistent with the retrieved data.
- The successful live workflow completed in two model requests.

### Limitations

- All customer and policy data is synthetic.
- Knowledge-base retrieval uses simple keyword matching.
- Customer authentication and authorization are not implemented.
- The live demonstration supports one tool-selection round,
  rather than unrestricted autonomous execution.
- The workflow does not evaluate answers over a larger test dataset.

### Conclusion

The experiment demonstrates a complete model-guided
tool-calling cycle: Gemini selects retrieval tools, Python
executes them, and Gemini incorporates the results into
a grounded customer-support response.
